<div style="background: linear-gradient(135deg, #0b4ea2, #ee1c25); padding: 20px; border-radius: 10px;">
  <h1 style="color: white; text-align: center; margin:0;">🇸🇰 ABM Migration — Slovakia</h1>
  <p style="color: white; text-align: center;">Interactive Agent-Based Simulation</p>
</div>

Configure the parameters in the panel below and click **«Run Simulation»**.

> 💡 **Tip:** To view the source code, click the arrow (▶) next to each section title to expand it. The code is hidden by default so you see only the interface.</VSCode.Cell>

In [ ]:
# @title 1. Setup & Installation (click to expand)
# @markdown Run this cell first to clone the repository and install dependencies.

!git clone https://github.com/SlovakZar/SlovakABM.git 2>/dev/null || echo "Repository already exists"
%cd SlovakABM/
%pip install matplotlib --quiet

from IPython.display import HTML, display
display(HTML('<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css">'))
print("Setup complete. Now run the next cell to launch the simulation.")

In [ ]:
# @title 🎛️ 2. Control Panel &amp; Run (click ▶ to expand)
# @markdown Configure simulation parameters, report sections, and scenario events below.
# @markdown Then click the **🚀 Run Simulation** button at the bottom.

import ipywidgets as widgets
from IPython.display import display, clear_output
import json, os
from pathlib import Path
from run import run

# ---------- styles ----------
style = {"description_width": "initial"}
layout_slider = widgets.Layout(width='80%')
layout_checkbox = widgets.Layout(width='auto')

# ========== TAB 1: PARAMETERS ==========
n_agents_w = widgets.IntSlider(
    value=50000, min=5000, max=70000, step=5000,
    description='<i class="fas fa-user-friends"></i> Agents:',
    style=style, layout=layout_slider
)
n_ticks_w = widgets.IntSlider(
    value=16, min=6, max=120, step=6,
    description='<i class="fas fa-clock"></i> Ticks:',
    style=style, layout=layout_slider
)
seed_w = widgets.IntText(value=1170, description='<i class="fas fa-dice"></i> Seed:', style=style)
params_box = widgets.VBox([n_agents_w, n_ticks_w, seed_w])

# ========== TAB 2: REPORT SECTIONS ==========
section_cbs = {
    "agent_params": widgets.Checkbox(
        value=True, description='<i class="fas fa-table"></i> Agent Parameters Matrix',
        style=style, layout=layout_checkbox
    ),
    "demographic": widgets.Checkbox(
        value=True, description='<i class="fas fa-users"></i> Demographic Portrait',
        style=style, layout=layout_checkbox
    ),
    "migration_trends": widgets.Checkbox(
        value=True, description='<i class="fas fa-chart-line"></i> Dynamics Summary &amp; Trends',
        style=style, layout=layout_checkbox
    ),
    "region_balance": widgets.Checkbox(
        value=True, description='<i class="fas fa-balance-scale"></i> Interregional Balance',
        style=style, layout=layout_checkbox
    ),
    "master_table": widgets.Checkbox(
        value=True, description='<i class="fas fa-list-alt"></i> Master Table (79 Districts)',
        style=style, layout=layout_checkbox
    ),
    "top_routes": widgets.Checkbox(
        value=True, description='<i class="fas fa-route"></i> Top-10 Moves &amp; Commutes',
        style=style, layout=layout_checkbox
    ),
    "heatmap": widgets.Checkbox(
        value=True, description='<i class="fas fa-map"></i> District Heatmap',
        style=style, layout=layout_checkbox
    ),
    "behavior_audit": widgets.Checkbox(
        value=False, description='<i class="fas fa-search"></i> Behavioral Audit',
        style=style, layout=layout_checkbox
    ),
}
sections_box = widgets.VBox(list(section_cbs.values()))

# ========== TAB 3: SCENARIO ==========
_ALL_DISTRICTS = [
    "District of Bánovce nad Bebravou", "District of Banská Bystrica",
    "District of Banská Štiavnica", "District of Bardejov",
    "District of Bratislava I", "District of Bratislava II",
    "District of Bratislava III", "District of Bratislava IV",
    "District of Bratislava V", "District of Brezno", "District of Bytča",
    "District of Čadca", "District of Detva", "District of Dolný Kubín",
    "District of Dunajská Streda", "District of Galanta", "District of Gelnica",
    "District of Hlohovec", "District of Humenné", "District of Ilava",
    "District of Kežmarok", "District of Komárno", "District of Košice I",
    "District of Košice II", "District of Košice III", "District of Košice IV",
    "District of Košice - okolie", "District of Krupina",
    "District of Kysucké Nové Mesto", "District of Levoča", "District of Levice",
    "District of Liptovský Mikuláš", "District of Lučenec", "District of Malacky",
    "District of Martin", "District of Medzilaborce", "District of Michalovce",
    "District of Myjava", "District of Námestovo", "District of Nitra",
    "District of Nové Mesto nad Váhom", "District of Nové Zámky",
    "District of Partizánske", "District of Pezinok", "District of Piešťany",
    "District of Poltár", "District of Poprad", "District of Považská Bystrica",
    "District of Prešov", "District of Prievidza", "District of Púchov",
    "District of Revúca", "District of Rimavská Sobota", "District of Rožňava",
    "District of Ružomberok", "District of Sabinov", "District of Senec",
    "District of Senica", "District of Skalica", "District of Snina",
    "District of Sobrance", "District of Spišská Nová Ves",
    "District of Stará Ľubovňa", "District of Stropkov", "District of Šaľa",
    "District of Topoľčany", "District of Trebišov", "District of Trenčín",
    "District of Trnava", "District of Turčianske Teplice", "District of Tvrdošín",
    "District of Veľký Krtíš", "District of Vranov nad Topľou",
    "District of Zlaté Moravce", "District of Zvolen", "District of Žarnovica",
    "District of Žiar nad Hronom", "District of Žilina",
]
_ALL_INDUSTRIES = [
    "Accommodation and food service activities",
    "Administrative and support service activities",
    "Agriculture, forestry and fishing",
    "Construction",
    "Human health and social work activities",
    "Information and communication",
    "Manufacturing total",
    "Other",
    "Professional, scientific and technical activities",
    "Public administration and defence",
    "Transportation and storage",
    "Water supply; sewerage, waste management and remediation activities",
    "Wholesale and retail trade; repair of motor vehicles and motorcycles",
]

# v5: size to number of jobs mapping
_SIZE_JOBS = {"small": 25, "medium": 130, "big": 400}

scenario_events = []

ev_type_w = widgets.Dropdown(
    options=["NEW_EMPLOYER", "CLOSED_EMPLOYER",
             "HOUSING_SHOCK", "NEW_INFRA", "CLOSED_INFRA"],
    value="NEW_EMPLOYER", description='Type:', style=style,
)
ev_district_w = widgets.Dropdown(
    options=_ALL_DISTRICTS, value="District of Žilina",
    description='District:', style=style,
)
ev_industry_w = widgets.Dropdown(
    options=_ALL_INDUSTRIES, value="Manufacturing total",
    description='Industry:', style=style,
)
ev_tick_w = widgets.IntSlider(value=6, min=1, max=120, description='Tick:', style=style)
ev_mag_w = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                description='Magnitude:', style=style)
ev_size_w = widgets.Dropdown(
    options=[("small (25 jobs)", "small"),
             ("medium (130 jobs)", "medium"), ("big (400 jobs)", "big")],
    value="small", description='Size:', style=style,
)
ev_n_agents_w = widgets.IntText(
    value=0, description='Layoffs (n):', style=style,
)
btn_add = widgets.Button(description="➕ Add", button_style="primary")
btn_clear = widgets.Button(description="🗑 Clear", button_style="danger")
ev_output = widgets.Output()

def _render_events():
    ev_output.clear_output(wait=True)
    with ev_output:
        if not scenario_events:
            display(widgets.HTML("<i style='color:gray'>No events yet</i>"))
        else:
            rows = ""
            for i, e in enumerate(scenario_events, 1):
                sz = e.get('size')
                if sz:
                    n_jobs = _SIZE_JOBS.get(sz, 0)
                    sz_display = f"{sz} ({n_jobs} jobs)"
                else:
                    sz_display = "—"
                mag_display = f"{e.get('magnitude', 0):.2f}" if e.get('magnitude') else "—"
                n_display = str(e.get('n_agents', 0)) if e.get('n_agents', 0) > 0 else "—"
                rows += f"<tr><td>{i}</td><td>{e['tick']}</td><td>{e['type']}</td>"
                rows += f"<td>{e['district']}</td><td>{e.get('industry','—')}</td>"
                rows += f"<td>{sz_display}</td><td>{n_display}</td><td>{mag_display}</td></tr>"
            display(widgets.HTML(
                "<table style='font-size:13px;border-collapse:collapse'>"
                "<tr style='background:#eee'><th>#</th><th>Tick</th><th>Type</th>"
                "<th>District</th><th>Industry</th><th>Size</th><th>n_agents</th><th>Magn.</th></tr>"
                + rows + "</table>"
            ))

def _on_add(b):
    ev_type = ev_type_w.value
    ev = {
        "tick": ev_tick_w.value,
        "type": ev_type,
        "district": ev_district_w.value,
    }
    # Industry — only for employer events
    if ev_type in ("NEW_EMPLOYER", "CLOSED_EMPLOYER"):
        ev["industry"] = ev_industry_w.value
        ev["size"] = ev_size_w.value
        ev["n_agents"] = ev_n_agents_w.value
    # Magnitude — for HOUSING_SHOCK and infra events
    if ev_type in ("HOUSING_SHOCK", "NEW_INFRA", "CLOSED_INFRA"):
        ev["magnitude"] = ev_mag_w.value
    scenario_events.append(ev)
    _render_events()

btn_add.on_click(_on_add)
btn_clear.on_click(lambda b: (scenario_events.clear(), _render_events()))

scenario_box = widgets.VBox([
    widgets.HBox([ev_type_w, ev_district_w, ev_industry_w]),
    widgets.HBox([ev_tick_w, ev_size_w, ev_n_agents_w, ev_mag_w]),
    widgets.HBox([btn_add, btn_clear]),
    ev_output
])
_render_events()

# ========== TABS ==========
tab = widgets.Tab()
tab.children = [params_box, sections_box, scenario_box]
tab.set_title(0, '⚙️ Parameters')
tab.set_title(1, '📊 Report')
tab.set_title(2, '📜 Scenario')

# ========== RUN BUTTON ==========
run_btn = widgets.Button(
    description='🚀 Run Simulation',
    button_style='success',
    layout=widgets.Layout(width='50%')
)
run_output = widgets.Output()

def on_run_clicked(b):
    with run_output:
        clear_output(wait=True)
        print("⏳ Simulation starting...")
        sections = {key: cb.value for key, cb in section_cbs.items()}
        scenario_path = Path("scenario.json")
        if scenario_events:
            scenario_path.write_text(
                json.dumps(scenario_events, ensure_ascii=False, indent=2),
                encoding="utf-8"
            )
        else:
            scenario_path.write_text("[]", encoding="utf-8")
        try:
            df_final, snapshots, tick_stats, all_action_log = run(
                n_agents=n_agents_w.value,
                n_ticks=n_ticks_w.value,
                seed=seed_w.value,
                detail=True,
                sections=sections,
            )
            print("✅ Simulation complete.")
            from IPython.display import Image as IPImage
            try:
                display(IPImage("heatmap.png"))
            except FileNotFoundError:
                print("⚠️ Heatmap not found.")
        except Exception as e:
            print(f"❌ Error: {e}")
        finally:
            if scenario_path.exists():
                scenario_path.unlink()

run_btn.on_click(on_run_clicked)

# ========== DISPLAY ==========
display(widgets.VBox([
    tab,
    widgets.HBox([run_btn], layout=widgets.Layout(justify_content='center')),
    run_output
]))